# RAGInGoa — Corpus Exploration

Voice → RAG → Answer. This notebook walks the offline pipeline: load the curated Goa corpus, chunk it, embed, build the dev vector index, retrieve and eyeball the hits.

In [ ]:
import sys
sys.path.insert(0, '../..')
from rag.dataset.loader import read_data
from rag.dataset.cleaner import clean_text

In [ ]:
docs = read_data('rag/data/samples/sample_goa_docs.jsonl')
print(f'{len(docs)} documents loaded')
for d in docs[:3]:
    print('-', d['metadata']['title'], '|', clean_text(d['content'])[:60], '...')

In [ ]:
from rag.chunking.chunk_manager import ChunkManager
chunk_manager = ChunkManager('sentence', {'size': 350})
chunks = chunk_manager.split(docs)
print(chunk_manager.stats(chunks))

In [ ]:
from rag.embeddings.embedder import get_embedder
from rag.retrieval.retriever import Retriever
from rag.retrieval.retrieval_config import RetrievalConfig
from rag.vector_db.index import build_index, load_index

emb = get_embedder()
index = load_index('rag/vector_db/index', model_name=emb.model_name()) or build_index(
    emb, chunks, 'rag/vector_db/index')
retriever = Retriever(emb, index, RetrievalConfig(top_k=4))

In [ ]:
import pandas as pd  # noqa
for hit in retriever.retrieve('When is the best time to visit Palolem?'):
    print(f"{hit['score']:.3f}", '-', hit['text'][:90])

## Next steps

- Swap `get_embedder()` for `SentenceTransformerEmbedder` (`pip install sentence-transformers`).
- Point the `VECTOR_DB_ROUTER` at chromadb/milvus for scale.
- Run `rag/evaluation/retrieval_eval.py` and `rag/benchmarking/benchmark.py` for numbers.